In [1]:
!pip install ollama


In [ ]:
import json
import datasets
from datasets import load_dataset


meu_dataset = load_dataset('json', data_files={
    'train': '/home/cecilia/Documentos/PIBIC/Fase2/DADOS-LLMS/train (1).json',
    'validation': '/home/cecilia/Documentos/PIBIC/Fase2/DADOS-LLMS/val.json',
    'test': '/home/cecilia/Documentos/PIBIC/Fase2/DADOS-LLMS/test (1).json'
})


In [ ]:
from openai import OpenAI
import pandas as pd
from typing import List, Dict, Any
import time
from tqdm.notebook import tqdm
import os
import json
class OpenRouterBatchInference:
    def __init__(self, api_key: str, models: List[str], system_prompt: str): 
        self.client = OpenAI(
            base_url="",
            api_key='ollama'     
        )
        self.models = models
        self.system_prompt = system_prompt


    def _create_messages(self, user_prompt: str) -> List[Dict[str, str]]:
        
        return [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"Input: {user_prompt}\nOutput:"}
        ]

    def _query_model(self, model: str, user_prompt: str) -> str:
       
        completion = self.client.chat.completions.create(
            model=model,
            messages=self._create_messages(user_prompt),
            timeout=120,
            max_tokens=150
            )




        msg = completion.choices[0].message

        print("content:", repr(msg.content))

        if msg.content:
          return msg.content

        if hasattr(msg, 'reasoning') and msg.reasoning:
          return msg.reasoning

        return "ERRO: output vazio"

    def generate_outputs(self, dataset: pd.DataFrame, save_path: str) -> Dict[str, List[Dict[str, Any]]]: # Update the type hint

        all_outputs = {model: [] for model in self.models}

        if os.path.exists(save_path):
          with open(save_path, 'r', encoding='utf-8') as f:
            all_outputs = json.load(f)
            print("Progresso anterior carregado com sucesso!")

        print("Montando os índices já processados na memória... Aguarde.")
        indices_processados = {
            model: set(int(item["index"]) for item in all_outputs[model] if "index" in item)
            for model in self.models
        }

        loop_progresso = tqdm(dataset.index, desc="Processando dataset")

       
        for index in loop_progresso:
          data_point = dataset.loc[index]
          user_prompt = f"Input: {data_point['sentence']}"

          if all(int(index) in indices_processados[model] for model in self.models):
                continue

          for model in self.models:

            if int(index) in indices_processados[model]:
                    continue

            
            try:
              loop_progresso.set_postfix(modelo=model)
              output = self._query_model(model, user_prompt)

              all_outputs[model].append({
                  "index": int(index),
                  "input": data_point.to_dict(),
                  "output": output
              })

            except Exception as e:
              print(f"\n[Erro] no modelo {model} no índice {index}: {e}")
              all_outputs[model].append({
                  "index": int(index),
                  "input": data_point.to_dict(),
                  "output": "null|Error#API"
              })

          with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(all_outputs, f, ensure_ascii=False, indent=4)

        return all_outputs


In [9]:
df_teste = pd.DataFrame(meu_dataset['test'])
df_val = pd.DataFrame(meu_dataset['validation'])

In [ ]:
caminho_salvar = '/home/cecilia/Documentos/PIBIC/Fase2/Resultados_Modelos/resultado_gemma_parcial (1).json'



In [ ]:

api_key = 'ollama'
models = [
  'gemma3:27b'
]


In [ ]:
system_prompt = ("""
You are an expert NLP model specialized in Aspect-Based Sentiment Analysis (ABSA).
Your task is to extract aspect terms and their corresponding entity-attribute categories from the user's text based STRICTLY on the taxonomy provided below.

#### ALLOWED TAXONOMY (ENTITY#ATTRIBUTE) ####
You must ONLY use the categories listed below. Do not invent or use any other combination.

- HOTEL & ACCOMMODATION:
  Hotel#General, Hotel#Comfort, Hotel#Cleanliness, Hotel#Design_features,
  Rooms#General, Rooms#Comfort, Rooms#Cleanliness, Rooms#Design_features,
  Room_Amenities#GeneralPIBIC/Dados_2ª_fase, Room_Amenities#Design_features, Location#General

- RESTAURANT / FOOD & DRINKS:
  Restaurant#General, Restaurant#Miscellaneous, Restaurant#Prices,
  Food#Quality, Food#Style_Options, Drinks#Quality, Ambience#General

- COMPUTERS & HARDWARE:
  LAPTOP#GENERAL, LAPTOP#QUALITY, LAPTOP#PRICE, LAPTOP#DESIGN_FEATURES, LAPTOP#OPERATION_PERFORMANCE,
  DISPLAY#GENERAL, DISPLAY#QUALITY, DISPLAY#DESIGN_FEATURES, DISPLAY#OPERATION_PERFORMANCE,
  KEYBOARD#GENERAL, KEYBOARD#DESIGN_FEATURES, KEYBOARD#OPERATION_PERFORMANCE,
  BATTERY#GENERAL, BATTERY#OPERATION_PERFORMANCE,
  MULTIMEDIA_DEVICES#OPERATION_PERFORMANCE

- BOOKS & CONTENT:
  Book#General, Book#Quality, Book#Author, Content#Plot, Content#Characters

- CLOTHING & FOOTWEAR:
  Shoes#General, Shoes#Quality, Shoes#Size, Shoes#Looking,
  Clothing#General, Clothing#Quality, Clothing#Size,
  Top#General, Top#Quality,
  Bottom#General, Bottom#Quality, Bottom#Size, Bottom#Looking

- SERVICES:
  Service#General

#### CONSTRAINTS & FORMATTING RULES ####
1. For each aspect found, respond strictly in the format: aspect_term|ENTITY#ATTRIBUTE
2. Respond ONLY with the extracted pairs. No introduction, no explanation, no markdown, no extra text before or after.
3. If multiple aspects are present, separate them with a comma and a space (e.g., term1|ENTITY#ATTRIBUTE, term2|ENTITY#ATTRIBUTE).
4. Match the category casing EXACTLY as shown in the taxonomy (e.g., 'LAPTOP#GENERAL' in uppercase, 'Hotel#General' in mixed case).
5. Use 'NULL' ONLY when the text has absolutely no word or pronoun referring to the aspect.
6. If no aspects from the allowed taxonomy are found, respond with: None



#### REFERENCE EXAMPLES ####

Example 1:
Input: The staff is friendly.
Output: staff|Service#General

Example 2:
Input: Awesome place and an awesome host!
Output: place|Hotel#General, host|Service#General

Example 3:
Input: In my opinion she has become a page-turner author and I just couldn't put this book down.
Output: author|Book#Author, book|Book#General

Example 4:
Input: Delicious.
Output: null|Food#Quality

Example 5:
Input: this product seemed perfect for me.
Output: product|LAPTOP#GENERAL

Example 6:
Input: They have always been just a tad tight.
Output: They|Shoes#Size

Example 7:
Input: Going back soon ( who doesnt like $ 1 draft beer ).
Output: NULL|Restaurant#General
"""
)


inference = OpenRouterBatchInference(
    api_key=api_key,
    models=models,
    system_prompt=system_prompt
)


In [14]:
import regex as re
def calcular_metricas(gabarito_set_ou_str, ia_output):

    if ia_output is None or isinstance(ia_output, dict) or str(ia_output).startswith("ERRO"):
      return {"Precisao": 0.0, "Recall": 0.0, "F1": 0.0,"Acuracia": 0.0}

    gabarito_str = str(gabarito_set_ou_str).lower()
    ia_str = str(ia_output).lower()

    ia_str = re.sub(r'\bimplicit\b', 'null', ia_str)

    def extrair_pares(texto):
        for char in ["{", "}", "[", "]", "'", '"']:
            texto = texto.replace(char, "")

        pares = set()
        itens = texto.split(',')
        for item in itens:
            item = item.strip()
            if '|' in item:
                item = re.sub(r'\s+', '', item)

                pares.add(item)
        return pares

    set_gabarito = extrair_pares(gabarito_str)
    set_ia = extrair_pares(ia_str)

    if not set_gabarito and not set_ia:
        return {"Precisao": 1.0, "Recall": 1.0, "F1": 1.0, "Acuracia": 1.0}

    if not set_gabarito or not set_ia:
        return {"Precisao": 0.0, "Recall": 0.0, "F1": 0.0, "Acuracia": 0.0}

    acertos = set_gabarito.intersection(set_ia)
    n_acertos = len(acertos)

    recall = n_acertos / len(set_gabarito)
    precisao = n_acertos / len(set_ia) if len(set_ia) > 0 else 0
    f1 = (2 * precisao * recall) / (precisao + recall) if (precisao + recall) > 0 else 0


    uniao = set_gabarito.union(set_ia)
    acuracia = n_acertos / len(uniao) if len(uniao) > 0 else 0

    return {"Precisao": precisao, "Recall": recall, "F1": f1, "Acuracia": acuracia}

In [11]:
modelo = models[0]


In [ ]:
import json

caminho = '/home/cecilia/Documentos/PIBIC/Fase2/Resultados_Modelos/resultado_gemma_final.json'

with open(caminho) as f:
    outputs = json.load(f)

modelo = list(outputs.keys())[0]

validos_antes = len(outputs[modelo])

outputs[modelo] = [
    item for item in outputs[modelo]
    if not (item.get('index') >= 1101 and item.get('output') == 'null|Error#API')
    and item.get('index') != 1328
]

validos_depois = len(outputs[modelo])
print(f'Removidos: {validos_antes - validos_depois}')
print(f'Válidos restantes: {validos_depois}')

with open(caminho, 'w', encoding='utf-8') as f:
    json.dump(outputs, f, ensure_ascii=False, indent=4)

print('Arquivo limpo e salvo!')

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

outputs = inference.generate_outputs(df_teste, caminho_salvar)

for i, item in enumerate(outputs[modelo]):
    print(f"--- Item {i} ---")
    print("Output:", repr(item['output']))
    print()


item = outputs[modelo][0]
print("Output bruto:", repr(item['output']))
resultados_pibic_final = []



modelo = models[0]

for i, item inPIBIC/Dados_2ª_fase enumerate(outputs[modelo]):
    frase_original = item['input']['sentence']
    gabarito_oficial = item['input']['rotulos']
    predicao = item['output']

    metricas = calcular_metricas(gabarito_oficial, predicao)
    resultados_pibic_final.append({
        "Frase": frase_original,
        "IA": predicao,
        "Gabarito": gabarito_oficial,
        "Precisao": metricas["Precisao"],
        "Recall": metricas["Recall"],
        "F1": metricas["F1"],
        "Acuracia": metricas["Acuracia"]
    })


df_final = pd.DataFrame(resultados_pibic_final)
print("\n--- MÉDIAS FINAIS DO EXPERIMENTO ---")
print(df_final[['Precisao', 'Recall', 'F1', 'Acuracia']].mean())

In [ ]:
import json
import pandas as pd


caminho_salvar = '/home/cecilia/Documentos/PIBIC/Fase2/Resultados_Modelos/resultado_gemma_parcial (1).json'

with open(caminho_salvar, 'r', encoding='utf-8') as f:
    outputs_carregados = json.load(f)

modelo = models[0] 

resultados_pibic_final = []

for i, item in enumerate(outputs_carregados[modelo]):
    frase_original = item['input']['sentence']
    gabarito_oficial = item['input']['rotulos']
    predicao = item['output']

    metricas = calcular_metricas(gabarito_oficial, predicao)
    
    resultados_pibic_final.append({
        "Frase": frase_original,
        "IA": predicao,
        "Gabarito": gabarito_oficial,
        "Precisao": metricas["Precisao"],
        "Recall": metricas["Recall"],
        "F1": metricas["F1"],
        "Acuracia": metricas["Acuracia"]
    })

df_final = pd.DataFrame(resultados_pibic_final)
print("\n--- MÉDIAS FINAIS DO EXPERIMENTO ---")
print(df_final[['Precisao', 'Recall', 'F1', 'Acuracia']].mean())